# Chapter 43: Recommendation and Ranking

Synthetic NRG distributor orders demonstrate popularity, latent-factor ranking, and top-k evaluation.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0,str(Path.cwd().parents[1]/'src'))
from datasciencebook.recommendation import popularity_scores,latent_scores,rank_unseen,precision_recall_at_k,ndcg_at_k
print('Imports ready.')


Imports ready.


In [ ]:
rng=np.random.default_rng(43);users=80;items=12
user_f=rng.normal(size=(users,3));item_f=rng.normal(size=(items,3));affinity=user_f@item_f.T
interactions=np.where(affinity+rng.normal(0,.8,(users,items))>1.0,rng.integers(1,6,(users,items)),0).astype(float)
train=interactions.copy();test={};
for u in range(users):
 pos=np.flatnonzero(train[u]>0)
 if len(pos)>1:test[u]=int(pos[-1]);train[u,pos[-1]]=0
print(f'Distributors: {users}; products: {items}; held-out users: {len(test)}')


Distributors: 80; products: 12; held-out users: 72


In [ ]:
pop=popularity_scores(train);latent=latent_scores(train,3)
def evaluate(score_rows):
 ps=[];rs=[];ns=[]
 for u,target in test.items():
  ranked=rank_unseen(score_rows[u],train[u]>0,5);p,r=precision_recall_at_k({target},ranked,5);ps.append(p);rs.append(r);ns.append(ndcg_at_k({target},ranked,5))
 return np.mean(ps),np.mean(rs),np.mean(ns)
pop_rows=np.tile(pop,(users,1))
for name,s in [('popularity',pop_rows),('latent',latent)]:
 p,r,n=evaluate(s);print(f'{name}: precision@5={p:.3f} recall@5={r:.3f} ndcg@5={n:.3f}')


popularity: precision@5=0.061 recall@5=0.306 ndcg@5=0.197
latent: precision@5=0.150 recall@5=0.750 ndcg@5=0.385


In [ ]:
u=0;ranked=rank_unseen(latent[u],train[u]>0,5);print('Distributor 0 ranked products:',ranked.tolist());print('Scores:',[round(float(latent[u,i]),2) for i in ranked])


Distributor 0 ranked products: [0, 1, 6, 4, 7]
Scores: [1.03, 0.79, 0.45, 0.27, 0.25]


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,4));axes[0].bar(range(items),pop);axes[0].set(xlabel='Product',ylabel='Training interactions',title='Popularity concentration');axes[1].imshow(train,aspect='auto',cmap='Blues');axes[1].set(xlabel='Product',ylabel='Distributor',title='Sparse interaction matrix');fig.tight_layout();plt.show()


## Interpretation

Latent ranking retrieves more held-out products in this simulation. Offline relevance is incomplete, however, because unobserved products may never have been exposed. Production decisions also need availability, margin, diversity, safety, and controlled experiments.


In [ ]:
# Practice: rerank the top candidates with an inventory-availability constraint.
